# Session 3: how likely is the worst case?
**A. Real-world problem.** Session 2 added up the tolerances: safe, but every part must be
held very tight. Yet it is very unlikely that five independent parts all sit at their worst
limit together. Statistical stack-up uses that fact to allow looser, cheaper parts, and the
price is a small, known fraction of assemblies that fail. Today you compute that fraction,
check it by simulation, see how a drifting process breaks it, and allocate tolerances to five
parts at the lowest (illustrative) cost.

*Educational model.* Parts are independent and normally distributed. Real processes can be
skewed, correlated or drifting; the cost model is illustrative, not data from a factory.

In [ ]:
import itertools
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# the five-part assembly (mm): housing (+) and four stacked parts (-)
NOMINALS = np.array([52.0, 10.0, 12.0, 15.0, 13.0])
SIGNS = np.array([1, -1, -1, -1, -1])
GAP_NOMINAL = float(np.dot(SIGNS, NOMINALS))     # 2.00 mm
GAP_TOL = 0.30                                    # the gap must stay within 2.00 +- 0.30 mm
K_COST = np.array([3.0, 0.4, 1.0, 0.2, 0.6])      # illustrative cost coefficients, cost unit x mm
T_MIN = 0.05                                      # illustrative: no process here can hold better than +-0.05 mm
NSIGMA = 3.0                                      # a part tolerance t is +-3 sigma of its process

## Step 1: worst case versus root-sum-square (RSS)
A part tolerance +-t is taken as +-3 sigma, so `sigma_i = t_i / 3`. For independent parts the
variances add:

    sigma_gap = sqrt( sum sigma_i^2 )     so the +-3 sigma gap tolerance is  sqrt( sum t_i^2 )

Worst case adds the tolerances directly: `sum t_i`. RSS is never larger than the sum.

In [ ]:
def worst_case_tol(tols):
    return float(np.sum(tols))


def rss_tol(tols):
    """Gap tolerance (+-3 sigma) for independent parts: square root of the sum of squares."""
    return float(np.sqrt(np.sum(np.square(tols))))


def sigma_gap(tols):
    return rss_tol(tols) / NSIGMA


def corner_range(tols):
    """Lowest and highest gap over all 2^n corners (every part at a limit)."""
    gaps = []
    for pick in itertools.product((-1, 1), repeat=len(NOMINALS)):
        gaps.append(float(np.dot(SIGNS, NOMINALS + np.array(pick) * np.asarray(tols))))
    return min(gaps), max(gaps)


def simulate_gaps(tols, n=200_000, seed=7, drift=0.0):
    """Gap of n random assemblies. Each part ~ Normal(nominal, t/3). `drift` shifts every process
    centre by drift x sigma_i in the direction that makes the gap bigger (0 = centred)."""
    rng = np.random.default_rng(seed)
    tols = np.asarray(tols, dtype=float)
    sig = tols / NSIGMA
    centre = NOMINALS + drift * sig * SIGNS
    parts = rng.normal(centre, sig, size=(n, len(NOMINALS)))
    return parts @ SIGNS

## Step 2: capability and the fraction out of tolerance
With limits LSL and USL (the lower and upper specification limits) and a gap that is normal
with mean mu and spread sigma:

    Cp  = (USL - LSL) / (6 sigma)
    Cpk = min(USL - mu, mu - LSL) / (3 sigma)          (penalises an off-centre process)
    fraction out = Phi((LSL - mu)/sigma) + 1 - Phi((USL - mu)/sigma)

where Phi(z) = (1 + erf(z / sqrt 2)) / 2 is the normal cumulative distribution. A common
guideline is Cp >= 1.33, i.e. the limits are four sigma away.

In [ ]:
def phi(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def _need_sigma(sigma):
    if sigma <= 0:
        raise ValueError("sigma must be positive")


def cp(lsl, usl, sigma):
    _need_sigma(sigma)
    return (usl - lsl) / (6.0 * sigma)


def cpk(lsl, usl, mean, sigma):
    _need_sigma(sigma)
    return min(usl - mean, mean - lsl) / (3.0 * sigma)


def fraction_out(lsl, usl, mean, sigma):
    _need_sigma(sigma)
    return phi((lsl - mean) / sigma) + 1.0 - phi((usl - mean) / sigma)


def spec_limits():
    return GAP_NOMINAL - GAP_TOL, GAP_NOMINAL + GAP_TOL

## Step 3: illustrative cost and allocation
Tighter tolerances cost more. We *assume* cost_i = k_i / t_i (halve the tolerance, double the
cost). Minimising `sum k_i / t_i` while keeping the RSS gap tolerance at `T` gives (by a
Lagrange multiplier) `t_i` proportional to `k_i^(1/3)`. Under worst-case addition it is
`t_i` proportional to `sqrt(k_i)`. A part with an expensive process (large k) is given the
looser tolerance.

In [ ]:
def cost(tols, k=K_COST):
    return float(np.sum(k / np.asarray(tols)))


def rss_target(cp_required=4.0 / 3.0, gap_tol=GAP_TOL):
    """RSS gap tolerance that gives the required Cp when the process is centred."""
    return gap_tol / cp_required


def allocate_rss(total, k=K_COST):
    w = k ** (1.0 / 3.0)
    return total * w / math.sqrt(float(np.sum(w ** 2)))


def allocate_worst_case(total, k=K_COST):
    w = np.sqrt(k)
    return total * w / float(np.sum(w))


def makeable(tols, t_min=T_MIN):
    """True if no part needs a tolerance tighter than the process limit."""
    return bool(np.all(np.asarray(tols) >= t_min - 1e-12))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    lsl, usl = spec_limits()
    t_eq = np.full(5, rss_target() / math.sqrt(5))
    print(f"gap nominal {GAP_NOMINAL:.2f} mm, limits {lsl:.2f} to {usl:.2f} mm")
    print(f"equal tolerances +-{t_eq[0]:.4f} mm: worst case {worst_case_tol(t_eq):.3f}, RSS {rss_tol(t_eq):.3f}")
    lo, hi = corner_range(t_eq)
    print(f"corners: {lo:.3f} to {hi:.3f} mm")
    g = simulate_gaps(t_eq)
    print(f"simulation: mean {g.mean():.4f}, std {g.std():.4f} (analytic {sigma_gap(t_eq):.4f})")
    s = sigma_gap(t_eq)
    print(f"Cp {cp(lsl, usl, s):.3f}, Cpk {cpk(lsl, usl, GAP_NOMINAL, s):.3f}, fraction out {fraction_out(lsl, usl, GAP_NOMINAL, s):.2e}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(g, bins=80, density=True, alpha=0.6, label="simulated assemblies")
    zz = np.linspace(lsl - 0.1, usl + 0.1, 300)
    ax.plot(zz, np.exp(-0.5 * ((zz - GAP_NOMINAL) / s) ** 2) / (s * math.sqrt(2 * math.pi)), "k", label="normal, sigma = RSS")
    ax.axvline(lsl, color="red", ls="--", label="limits")
    ax.axvline(usl, color="red", ls="--")
    ax.set_xlabel("assembly gap (mm)")
    ax.set_ylabel("probability density (1/mm)")
    ax.set_title("Five-part stack: simulated gap versus the RSS normal curve")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "gap_histogram.png", dpi=120)

    t_rss = allocate_rss(rss_target())
    t_wc = allocate_worst_case(GAP_TOL)
    print("RSS allocation  :", np.round(t_rss, 4), f"cost {cost(t_rss):.2f}, makeable {makeable(t_rss)}")
    print("equal (RSS)     :", np.round(t_eq, 4), f"cost {cost(t_eq):.2f}")
    print("worst-case alloc:", np.round(t_wc, 4), f"cost {cost(t_wc):.2f}, makeable {makeable(t_wc)}")
    t_wc_eq = np.full(5, GAP_TOL / 5)
    print(f"worst-case equal +-{t_wc_eq[0]:.3f}: cost {cost(t_wc_eq):.2f}, makeable {makeable(t_wc_eq)}")

    fig2, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    labels = ["worst case\n(sum)", "RSS\n(statistical)"]
    axes[0].bar(labels, [worst_case_tol(t_eq), rss_tol(t_eq)], color=["tab:red", "tab:green"])
    axes[0].set_ylabel("gap tolerance, plus or minus (mm)")
    axes[0].set_title("Same five parts, two ways to add")
    drifts = np.linspace(0, 2.0, 21)
    sim = [np.mean((simulate_gaps(t_rss, 100_000, 3, d) < lsl) | (simulate_gaps(t_rss, 100_000, 3, d) > usl)) for d in drifts]
    s_r = sigma_gap(t_rss)
    ana = [fraction_out(lsl, usl, GAP_NOMINAL + d * float(np.sum(t_rss / NSIGMA)), s_r) for d in drifts]
    axes[1].plot(drifts, 100 * np.array(ana), label="analytic (erf)")
    axes[1].plot(drifts, 100 * np.array(sim), "o", ms=3, label="simulation")
    axes[1].set_xlabel("process drift of every part (multiples of its sigma)")
    axes[1].set_ylabel("assemblies out of tolerance (%)")
    axes[1].set_title("A drifting process breaks the stack")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "worst_case_vs_rss.png", dpi=120)

    d1 = simulate_gaps(t_rss, 400_000, 11, 1.0)
    print(f"RSS allocation with every process drifted by 1 sigma: {np.mean((d1 < lsl) | (d1 > usl)):.4f} out (analytic "
          f"{fraction_out(lsl, usl, GAP_NOMINAL + float(np.sum(t_rss / NSIGMA)), s_r):.4f})")

## Where this stops
Real parts are not always normal, and processes drift. The roadmap shows how a control chart
watches for drift and how allowed tolerance grades are chosen from a standard list.

## Coding tasks
1. **Easy.** Use `simulate_gaps(t, 100000, 1, drift=1.0)` with equal tolerances of 0.10 mm and compute the fraction out with the limits from `spec_limits()`; compare with `fraction_out`.
2. **Stretch.** Change `K_COST` so that one part is 20 times as expensive to tighten as the others; run `allocate_rss` and describe how the tolerances change.
3. **Explain.** Explain why the cube root of k, not the square root, appears in the RSS allocation, and why the square root appears for worst case.

## Assessment
- **A. Conceptual.** Why is the root-sum-square tolerance never larger than the worst-case sum, and what assumption lets us use it? What do we pay for the smaller number?
- **B. Numerical.** A gap has limits 1.70 and 2.30 mm and a process with mean 2.10 mm and sigma 0.075 mm. Find Cp and Cpk and say what the difference means.
- **C. Graph and simulation interpretation.** In the drift graph, what happens to the fraction out when the drift grows from 0 to 1 sigma for the allocated tolerances? Read it at 0.5 and 1.0 sigma.
- **D. Design and optimisation challenge.** Design: five parts get equal tolerances. Find the equal tolerance t that gives Cp = 1.33 (four sigma) for a gap of 2.00 +-0.30 mm using RSS, and compare its cost, using k = 3.0, 0.4, 1.0, 0.2 and 0.6, with the equal worst-case design. Is the worst-case design makeable if no process can hold better than +-0.05 mm?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    lsl, usl = L.spec_limits()
    checks.append(("gap nominal is 2.00 mm and the limits are 1.70 to 2.30 mm",
                   abs(L.GAP_NOMINAL - 2.0) < 1e-9 and abs(lsl - 1.7) < 1e-9 and abs(usl - 2.3) < 1e-9))

    # worked example: five parts, +-0.10 mm each
    t = np.full(5, 0.10)
    wc, rs = L.worst_case_tol(t), L.rss_tol(t)
    sg = L.sigma_gap(t)
    checks.append((f"worked example: worst case 0.500, RSS 0.224 mm (got {wc:.3f}, {rs:.3f})",
                   abs(wc - 0.5) < 1e-9 and abs(rs - 0.2236) < 1e-4))
    checks.append(("worked example: sigma of the gap is 0.0745 mm", abs(sg - 0.0745) < 1e-4))
    checks.append(("worked example: Cp is 1.34 (above the 1.33 guideline)", abs(L.cp(lsl, usl, sg) - 1.3416) < 1e-3))
    checks.append(("worked example: fraction out is 5.7e-5 (0.0057 percent)", abs(L.fraction_out(lsl, usl, 2.0, sg) - 5.70e-5) < 1e-7))

    # validation against exact corner enumeration
    lo, hi = L.corner_range(t)
    checks.append(("corner enumeration (32 corners) gives exactly 2 +- the sum of tolerances", abs(lo - 1.5) < 1e-9 and abs(hi - 2.5) < 1e-9))
    rng = np.random.default_rng(3)
    ok = True
    for _ in range(10):
        tt = rng.uniform(0.02, 0.2, 5)
        a, b = L.corner_range(tt)
        ok &= abs((b - a) / 2 - L.worst_case_tol(tt)) < 1e-9 and L.rss_tol(tt) <= L.worst_case_tol(tt) + 1e-12
    checks.append(("RSS never exceeds the worst-case sum (10 random sets)", bool(ok)))

    # validation against Monte Carlo
    g = L.simulate_gaps(t, n=400_000, seed=5)
    checks.append((f"Monte Carlo sigma matches the RSS formula within 1 percent (got {g.std():.4f})", abs(g.std() / sg - 1) < 0.01))
    checks.append(("Monte Carlo mean is the nominal gap", abs(g.mean() - 2.0) < 5e-4))
    checks.append(("no simulated assembly leaves the worst-case corners", bool(g.min() > lo and g.max() < hi)))
    checks.append(("simulation is repeatable for the same seed", np.array_equal(L.simulate_gaps(t, 1000, 9), L.simulate_gaps(t, 1000, 9))))

    # Cp, Cpk and the erf fraction
    checks.append(("hand calculation: +-3 sigma limits give 0.27 percent out (erf)", abs(L.fraction_out(-3, 3, 0, 1) - 0.0026998) < 1e-6))
    sim = float(np.mean(np.abs(g - 2.0) > 3 * sg))
    checks.append((f"simulation agrees with erf at +-3 sigma within 0.05 percentage points (got {100 * sim:.3f}%)", abs(sim - 0.0027) < 5e-4))
    checks.append(("Cp = 1.333 means limits four sigma away: 6.3e-5 out", abs(L.fraction_out(1.7, 2.3, 2.0, 0.075) - 6.334e-5) < 1e-7
                   and abs(L.cp(1.7, 2.3, 0.075) - 4 / 3) < 1e-9))
    checks.append(("Cpk = Cp for a centred process", abs(L.cpk(1.7, 2.3, 2.0, 0.075) - L.cp(1.7, 2.3, 0.075)) < 1e-9))
    checks.append(("Cpk = 0.889 when the mean is 2.10 mm and sigma 0.075", abs(L.cpk(1.7, 2.3, 2.1, 0.075) - 0.8889) < 1e-3))
    try:
        L.cp(1.7, 2.3, 0.0)
        bad = False
    except ValueError:
        bad = True
    checks.append(("edge: sigma = 0 is rejected", bad))

    # allocation and cost
    target = L.rss_target()
    checks.append(("RSS target for Cp 1.333 is 0.225 mm", abs(target - 0.225) < 1e-9))
    tr = L.allocate_rss(target)
    checks.append(("allocated tolerances meet the RSS target exactly", abs(L.rss_tol(tr) - 0.225) < 1e-9))
    checks.append(("allocation: the cheapest-to-tighten part gets the tightest tolerance",
                   int(np.argmin(tr)) == int(np.argmin(L.K_COST)) and int(np.argmax(tr)) == int(np.argmax(L.K_COST))))
    t_eq = np.full(5, target / math.sqrt(5))
    checks.append((f"mini-project: cost {L.cost(tr):.2f} beats equal allocation {L.cost(t_eq):.2f} by 13 percent",
                   abs(1 - L.cost(tr) / L.cost(t_eq) - 0.1303) < 5e-4))
    # the closed form really is the minimum: random allocations with the same RSS never do better
    best = True
    for _ in range(2000):
        r = rng.uniform(0.2, 1.0, 5)
        r = r * target / np.sqrt(np.sum(r ** 2))
        best &= L.cost(r) >= L.cost(tr) - 1e-9
    checks.append(("no random allocation with the same RSS is cheaper (2000 trials)", bool(best)))
    checks.append(("all five allocated tolerances can be made (>= 0.05 mm)", L.makeable(tr) and abs(tr.min() - 0.0608) < 1e-4))
    checks.append(("equal allocation of identical k gives equal tolerances", np.allclose(L.allocate_rss(0.2, np.ones(5)), 0.2 / math.sqrt(5))))
    tw = L.allocate_worst_case(L.GAP_TOL)
    checks.append(("worst-case allocation sums to 0.30 but needs +-0.029 mm on one part: not makeable", abs(tw.sum() - 0.3) < 1e-9 and not L.makeable(tw)))
    checks.append(("worst-case equal allocation costs 86.67 against 44.94 for RSS",
                   abs(L.cost(np.full(5, 0.06)) - 86.667) < 1e-2 and abs(L.cost(tr) - 44.94) < 1e-2))

    # failure case 1: worst-case design too tight to be made
    t_wc20 = np.full(5, 0.20 / 5)
    t_rss20 = np.full(5, L.rss_target(gap_tol=0.20) / math.sqrt(5))
    checks.append(("failure case: for a +-0.20 gap, worst-case +-0.040 per part cannot be made but RSS +-0.067 can",
                   (not L.makeable(t_wc20)) and L.makeable(t_rss20) and abs(t_rss20[0] - 0.0671) < 1e-4))

    # failure case 2 and sensitivity: process centres drifted by one sigma
    s_r = L.sigma_gap(tr)
    shift = float(np.sum(tr / 3.0))
    ana = L.fraction_out(lsl, usl, 2.0 + shift, s_r)
    gd = L.simulate_gaps(tr, n=400_000, seed=11, drift=1.0)
    simd = float(np.mean((gd < lsl) | (gd > usl)))
    checks.append((f"failure case: a 1 sigma drift of every part makes 3.1 percent fail (erf {100 * ana:.2f}%, simulation {100 * simd:.2f}%)",
                   abs(ana - 0.0308) < 5e-4 and abs(simd - ana) < 2e-3))
    checks.append(("the centred design has Cp 1.333, but after the drift Cpk drops to 0.62",
                   abs(L.cp(lsl, usl, s_r) - 4 / 3) < 1e-9 and abs(L.cpk(lsl, usl, 2.0 + shift, s_r) - 0.623) < 1e-3))
    f = [L.fraction_out(lsl, usl, 2.0 + d * shift, s_r) for d in (0, 0.5, 1.0, 1.5)]
    checks.append(("sensitivity: out-of-tolerance fraction rises with drift (0.006, 0.17, 3.1, 21 percent)",
                   all(a < b for a, b in zip(f, f[1:])) and abs(f[1] - 0.00167) < 1e-5 and abs(f[3] - 0.2106) < 1e-3))
    # numbers used in the text and the mini-project
    checks.append(("allocation rounds to 0.150, 0.077, 0.104, 0.061, 0.088 mm", list(np.round(tr, 3)) == [0.150, 0.077, 0.104, 0.061, 0.088]))
    checks.append(("equal allocation is +-0.101 mm per part at cost 51.68", abs(t_eq[0] - 0.1006) < 1e-4 and abs(L.cost(t_eq) - 51.68) < 5e-3))
    checks.append(("sigma of the allocated gap is 0.075 mm", abs(s_r - 0.075) < 1e-9))
    checks.append(("app challenge: t = 0.09, drift 0.5 sigma gives Cpk about 1.12",
                   abs(L.cpk(lsl, usl, 2.0 + 0.5 * 5 * 0.09 / 3, math.sqrt(5) * 0.09 / 3) - 1.118) < 2e-3))
    checks.append(("a part toleranced +-0.10 costs half as much as +-0.05 with the same k (cost model)",
                   abs(L.cost(np.full(5, 0.05)) / L.cost(np.full(5, 0.10)) - 2.0) < 1e-9))
    checks.append(("worked example: limits are 4.02 sigma away and about 1 assembly in 17500 is out",
                   abs(0.3 / sg - 4.025) < 5e-3 and abs(1 / L.fraction_out(lsl, usl, 2.0, sg) - 17500) < 100))
    checks.append(("simulated drift fraction rounds to 0.0306 and the worst-case equal design costs 68 percent more",
                   abs(simd - 0.0306) < 5e-5 and abs(L.cost(np.full(5, 0.06)) / L.cost(t_eq) - 1.677) < 2e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
